# 03 The Association Between Exposure and Disease: 2×2 Tables and Inferential Statistics

In the Legionnaires' disease cluster at Pine and Cypress Nursing Home, the supervisor asks: "Do people who use the shower have a higher risk of infection?" A senior outbreak investigator follows up: "How do you use statistics to 'prove' it?"

In this lesson you'll learn: **the difference between description and inference → study design (cohort vs. case-control) → the 2×2 contingency table → the risk ratio (RR) → the odds ratio (OR) → the confidence interval (CI) → chi-square / Fisher tests → multi-factor summary and forest plot**.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Descriptive vs. Inferential Statistics: What Are We Doing?

In Ch02 we used **descriptive statistics** (means, frequency distributions, charts) to summarize what the data looked like. But the supervisor's question is:

> "Is there **an association** between shower use and infection?"

That calls for **inferential statistics**—using sample data to infer whether an association truly exists in the population, or whether the difference we see is just the result of **chance (random error)**.

### Key Concepts

- **Null hypothesis (H₀)**: shower use and infection are independent (no association)
- **Alternative hypothesis (H₁)**: shower use and infection are associated
- **p-value**: the probability of observing the current data (or something more extreme) assuming H₀ is true. The smaller the p, the more reason to reject H₀
- **Confidence interval (CI)**: a plausible range for the effect measure. If the 95% CI does not include the "no-effect value" (RR=1 or OR=1), then p < 0.05

### Study Design Determines What You Can Compute—Four Common Designs

#### ❶ Cohort Study—🎬 A Following-Along Documentary
First split into two groups by exposure (got rained on vs. didn't), then follow along to see who catches a cold later. You know everyone → **complete denominators → compute RR**.

#### ❷ Case-Control Study—🕵️ Detective Work
First find the sick people (cases), then pick people who aren't sick (controls), and look back at their exposure history. The number of controls is decided by you → **no complete denominators → can only compute OR**. Good for: rare diseases, large-scale vaccine effectiveness surveillance.

#### ❸ Nested Case-Control—🏠 Reviewing Surveillance Footage
A cohort is already being followed, but testing everyone one by one is too expensive. Once someone develops disease, pull cases + controls from the cohort to test. Combines the representativeness of a cohort with the efficiency of case-control.

#### ❹ Matched Case-Control—👯 A Twin Experiment
Match each case with a control of similar age and sex, so confounders "cancel out." The analysis must use conditional logistic regression.

| Study design | Sampling approach | Measure available | Analogy |
|---------|---------|--------|------|
| Cohort study | Group by exposure, follow for outcome | **RR** | 🎬 Following-along documentary |
| Case-control | Group by disease, look back at exposure | **OR** | 🕵️ Detective work |
| Nested case-control | Pick cases + controls from a cohort | **OR** | 🏠 Reviewing footage |
| Matched case-control | 1:n matching, control confounders | **OR** (conditional) | 👯 Twin experiment |

> 🎬 **This investigation** = a retrospective cohort study: all 280 residents are included, and both exposure and outcome are known → you can directly compute the **RR**.

In [ ]:
# --- Step 1: Data preparation ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency, fisher_exact
from epi_learning.metrics import risk_ratio, odds_ratio

# -- CJK font setup (prevents Chinese labels showing as boxes □□□) --
# matplotlib only recognizes English fonts by default, so Chinese characters turn into "tofu blocks"
# Fix: manually scan the system font directories and register all CJK (Chinese/Japanese/Korean) fonts
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# Dynamically detect the actually registered CJK font names (avoids the .ttc face 0 trap)
_discovered = []
for _entry in fm.fontManager.ttflist:
    _nlower = _entry.name.lower()
    if any(_kw in _nlower for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
_cjk_fonts = list(_discovered)
for _n in _preferred:
    if _n not in _cjk_fonts:
        _cjk_fonts.append(_n)
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Create a binary "infected" column (0/1)
# clinical_severity == "not_ill" means no symptoms and not infected; everything else counts as infected
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

print(f"Total: {len(df)} people, infected: {df['infected'].sum()} people")
print(f"Overall attack rate: {df['infected'].mean():.1%}")

## Step 2: Build the 2×2 Table

We already have 280 residents and an `infected` column. Now we'll organize it into a **2×2 contingency table**—the fundamental data structure for epidemiological association analysis.

> **💡 Tip**: with `pd.crosstab()`, the first argument becomes the "rows" and the second becomes the "columns." When extracting the four cells a/b/c/d, rename them with labels first so you don't have to remember which is 0 and which is 1.

In [ ]:
# --- Step 2: Build the 2x2 table (Shower x Infection) ---
# The first argument to pd.crosstab() → rows, the second → columns
# margins=True automatically adds a subtotal row and a subtotal column
ct_shower = pd.crosstab(
    df["shower_use"], df["infected"],
    margins=True, margins_name="Total",
)
# Rename for readability (the raw 0/1 values aren't intuitive)
ct_shower.index = ["No shower", "Shower", "Total"]
ct_shower.columns = ["Not infected", "Infected", "Total"]
print(ct_shower)

# ── Extract the four cells of the 2×2 table ──
# crosstab's row order follows the sort order of the raw values (0 before 1)
# After renaming, we extract by the renamed labels so we don't have to remember which is 0 and which is 1
a = int(ct_shower.loc["Shower", "Infected"])        # a = exposed + infected
b = int(ct_shower.loc["Shower", "Not infected"])      # b = exposed + not infected
c = int(ct_shower.loc["No shower", "Infected"])      # c = unexposed + infected
d = int(ct_shower.loc["No shower", "Not infected"])    # d = unexposed + not infected

# Compute the attack rate for each group separately
print(f"\nExposed group (Shower) attack rate: {a/(a+b):.1%}")
print(f"Unexposed group (No shower) attack rate: {c/(c+d):.1%}")

In [ ]:
# --- Step 3: Compute the Risk Ratio ---
rr = risk_ratio(a, a + b, c, c + d)
print(f"Shower use -> infection RR = {rr:.3f}")
print(f"  Interpretation: shower users' infection risk is {rr:.1f}x that of non-users")
print(f"  RR = 1 -> no association | RR > 1 -> exposure may increase risk | RR < 1 -> may be protective")

## RR vs OR — What's the Difference Between Risk and Odds?

### RR in Plain Language ⚖️

RR = 2 means "exposed people have twice the **risk** of disease as unexposed people." Like weighing on a scale: put the exposed group's risk on the left and the unexposed group's on the right.

### OR in Plain Language 🎰

OR = 3 means "exposed people have 3 times the **odds** of disease as unexposed people." Note: this is about "odds," not "risk"!

- **Risk = p**: 30 of 100 people get sick → 30/100 = 0.3 (divide by **everyone**)
- **Odds = p/(1−p)**: 30 sick vs 70 not sick → 30/70 ≈ 0.43 (sick ÷ **not sick**)

### Three Important Ideas

1. **For rare diseases, OR ≈ RR** (the 10% rule): when p is small, 1−p ≈ 1, so odds ≈ risk → OR ≈ RR.
   🍬 Of 100 candies, 3 are sour: risk = 3/100 = 3%, odds = 3/97 ≈ 3.1%—practically the same.

2. **When the attack rate is high, OR is systematically larger than RR**: this outbreak's attack rate is ~43%, so the OR overestimates the RR by 30-50%! You can't use the OR to say "risk is X times higher."

3. **OR is the native output of logistic regression**: the model computes log(odds), so exp(β) = OR. Ch06 will cover this.

### How Do You Estimate Vaccine Effectiveness (VE)?

- **Cohort study**: VE = 1 − RR (e.g., RR = 0.2 → VE = 80%)
- **Case-control**: VE = 1 − OR (≈ 1 − RR when the disease is rare)
- Many COVID-19 vaccine effectiveness figures were computed with a test-negative case-control design

In [ ]:
# --- Step 4: Compute the Odds Ratio ---
# odds = p / (1-p), which is different from risk = p
# OR = (a × d) / (b × c)
or_val = odds_ratio(a, b, c, d)
print(f"Shower use -> infection OR = {or_val:.3f}")
print(f"  (compared with RR = {rr:.3f})")
print(f"\nThis dataset's attack rate = {df['infected'].mean():.1%} (not a rare disease)")
print(f"-> OR ({or_val:.3f}) is greater than RR ({rr:.3f}), which is expected")
print(f"-> When disease is rare, OR ~= RR; the higher the attack rate, the more OR deviates from RR")
print(f"\n[!] This case's attack rate ~43%, so don't use OR to say 'how many times the risk'!")
print(f"   Correct: shower users' infection risk is {rr:.1f}x that of non-users (RR)")
print(f"   Wrong: shower users' infection risk is {or_val:.1f}x that of non-users (OR) <- overestimated!")

## Step 5: 95% Confidence Interval — Why Take the log First?

The CI is the part that makes beginners' heads spin the most. The key intuition:

1. **The original scale is asymmetric**: the range of RR/OR is 0 to ∞, centered on 1, with only a short stretch on the left (0 to 1) but extending to infinity on the right
2. **Log-transform to a symmetric scale**: after taking ln(), the scale becomes -∞ to +∞, centered on 0, where you can use the normal distribution's ±1.96×SE
3. **exp back**: convert the log-scale lower and upper bounds back to the original scale with exp(), and that's the 95% CI

In [ ]:
# --- Step 5: 95% confidence intervals (RR and OR) ---

# ── 95% CI for RR: Katz method ──

# (a) Take the natural log: move RR from the asymmetric scale (0, ∞) to the symmetric scale (-∞, +∞)
ln_rr = np.log(rr)

# (b) Compute the standard error (SE): a measure of how precise the ln(RR) estimate is
#     The formula comes from Katz (1978), derived from the four cells of the 2×2 table
se_ln_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))

# (c) On the log scale, ±1.96 × SE (1.96 is the z-value for 95% of the normal distribution)
# (d) Use exp() to convert back to the original scale → get the lower and upper bounds of the CI
ci_rr_lo = np.exp(ln_rr - 1.96 * se_ln_rr)
ci_rr_hi = np.exp(ln_rr + 1.96 * se_ln_rr)

# ── 95% CI for OR: Woolf method ──
# Same principle, only the SE formula differs (using the sum of the reciprocals of a, b, c, d)
ln_or = np.log(or_val)
se_ln_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_or_lo = np.exp(ln_or - 1.96 * se_ln_or)
ci_or_hi = np.exp(ln_or + 1.96 * se_ln_or)

print("=== 95% Confidence Interval Comparison ===")
print(f"RR = {rr:.3f} (95% CI: {ci_rr_lo:.3f} – {ci_rr_hi:.3f})")
print(f"OR = {or_val:.3f} (95% CI: {ci_or_lo:.3f} – {ci_or_hi:.3f})")
sig_rr = "significant" if ci_rr_lo > 1 else "not significant"
sig_or = "significant" if ci_or_lo > 1 else "not significant"
print(f"\nRR's CI {'does not ' if ci_rr_lo <= 1 else ''}contain 1 -> {sig_rr}")
print(f"OR's CI {'does not ' if ci_or_lo <= 1 else ''}contain 1 -> {sig_or}")

## Step 6: The Chi-Square Test

The core logic of the chi-square test: if exposure and infection really were "unrelated" (H₀ is true), how many people should we observe in each cell? How far are the actual numbers from that expectation?

- **Small gap** → small χ² → large p → not significant (observed values are close to what H₀ predicts)
- **Large gap** → large χ² → small p → significant! (observed values are far from what H₀ predicts)

In [ ]:
# --- Step 6: Chi-square test ---
# H₀: shower use and infection are independent (no association)
# Logic: compare the "observed counts" with the "expected counts assuming no association"
contingency = [[a, b], [c, d]]
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p:.4f}")
print(f"\nExpected-value table (expected counts when H₀ is true):")
print(pd.DataFrame(
    expected.round(1),
    index=["Shower", "No shower"],
    columns=["Infected", "Not infected"],
))

# Check whether all expected values are >= 5
min_expected = expected.min()
print(f"\nMinimum expected value = {min_expected:.1f}", end="")
if min_expected >= 5:
    print(" -> meets the chi-square test assumption")
else:
    print(" -> < 5, recommend using Fisher's exact test instead")

In [ ]:
# --- Step 7: Fisher's exact test ---
# An alternative when expected values are < 5 (more precise for small samples)
oddsr_fisher, p_fisher = fisher_exact(contingency)
print(f"Fisher's exact test:")
print(f"  OR = {oddsr_fisher:.3f}")
print(f"  p-value = {p_fisher:.4f}")
print(f"\nChi-square test p = {p:.4f} vs Fisher p = {p_fisher:.4f}")
print("(This example's sample is large enough that the two tests agree closely; the difference is more pronounced with small samples)")

In [ ]:
# --- Step 8: Second exposure factor -- Hydrotherapy use ---
# Apply the same workflow to a second exposure factor (exactly the same as Steps 2–6, just a different exposure variable)
# Step 9 will use a loop to automate this process, so no more copy-pasting by hand
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a2, b2 = int(ct_hydro.loc[1, 1]), int(ct_hydro.loc[1, 0])  # exposed+infected, exposed+not infected
c2, d2 = int(ct_hydro.loc[0, 1]), int(ct_hydro.loc[0, 0])  # unexposed+infected, unexposed+not infected

# Effect measures
rr2 = risk_ratio(a2, a2 + b2, c2, c2 + d2)
or2 = odds_ratio(a2, b2, c2, d2)
chi2_2, p2, _, _ = chi2_contingency([[a2, b2], [c2, d2]])

# RR CI (Katz method, same as Step 5)
ln_rr2 = np.log(rr2)
se_rr2 = np.sqrt(1/a2 - 1/(a2+b2) + 1/c2 - 1/(c2+d2))
ci_rr2_lo = np.exp(ln_rr2 - 1.96 * se_rr2)
ci_rr2_hi = np.exp(ln_rr2 + 1.96 * se_rr2)

# OR CI (Woolf method, same as Step 5)
ln_or2 = np.log(or2)
se_or2 = np.sqrt(1/a2 + 1/b2 + 1/c2 + 1/d2)
ci_or2_lo = np.exp(ln_or2 - 1.96 * se_or2)
ci_or2_hi = np.exp(ln_or2 + 1.96 * se_or2)

print("Hydrotherapy use -> infection")
print(f"  RR = {rr2:.3f} (95% CI: {ci_rr2_lo:.3f} – {ci_rr2_hi:.3f})")
print(f"  OR = {or2:.3f} (95% CI: {ci_or2_lo:.3f} – {ci_or2_hi:.3f})")
print(f"  chi-square p-value = {p2:.4f}")

## What Is a Forest Plot?

The **forest plot** is one of the most common charts in epidemiology and evidence-based medicine, often used in systematic reviews and meta-analyses, but it's also very useful in outbreak investigations—it lets you **compare the effect sizes and statistical significance of multiple exposure factors at a glance**.

### How Do You Read a Forest Plot?

- **Dot (●)**: the point estimate (RR in this example)
- **Horizontal line segment (─)**: the 95% confidence interval
- **Dashed line (RR = 1)**: the no-effect line. A CI that crosses the dashed line = not significant; a CI entirely to the right of the dashed line = exposure significantly increases risk

Next we'll draw a forest plot to compare the crude RRs of 8 risk factors at once.

In [ ]:
# --- Step 9: Multi-factor crude effect-measure summary table + forest plot ---
# In a real outbreak investigation you won't just look at one or two factors.
# The loop below systematically applies Steps 2–6 to every candidate exposure.

# List all exposure factors to test (binary 0/1 variables)
factors = [
    "shower_use", "hydrotherapy_use",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
]
# Convert "ever smoked" into a binary variable
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)
factors.append("ever_smoker")

# ── Loop: repeat Steps 2–6 for each factor ──
# Each iteration does 5 things: (a) build the 2×2 table → (b) compute RR/OR → (c) compute the CI → (d) chi-square test → (e) store the results
results = []
for factor in factors:
    # (a) Build the 2×2 table, extract a, b, c, d
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])   # exposed + infected
    b_i = int(ct.loc[1, 0])   # exposed + not infected
    c_i = int(ct.loc[0, 1])   # unexposed + infected
    d_i = int(ct.loc[0, 0])   # unexposed + not infected

    # (b) Effect measures: RR and OR
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    or_i = odds_ratio(a_i, b_i, c_i, d_i)

    # (c) 95% CI for RR (Katz method, same as Step 5)
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)

    # (d) Chi-square test
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])

    # (e) Store this factor's results
    results.append({
        "factor": factor,
        "RR": round(rr_i, 3),
        "CI_lower": round(ci_lo, 3),
        "CI_upper": round(ci_hi, 3),
        "OR": round(or_i, 3),
        "p-value": round(p_i, 4),
    })

# Assemble into a table, sorted by RR from high to low (most suspicious factors first)
rr_table = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== Multi-Factor Crude Effect-Measure Summary Table ===")
display_df = rr_table.copy()
display_df["95% CI"] = display_df.apply(
    lambda r: f"{r['CI_lower']:.3f}–{r['CI_upper']:.3f}", axis=1
)
print(display_df[["factor", "RR", "95% CI", "OR", "p-value"]].to_string(index=False))

# --- Forest Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
rr_sorted = rr_table.reset_index(drop=True)
y_pos = range(len(rr_sorted))
ax.errorbar(
    rr_sorted["RR"], y_pos,
    xerr=[rr_sorted["RR"] - rr_sorted["CI_lower"],
          rr_sorted["CI_upper"] - rr_sorted["RR"]],
    fmt="o", color="#D97757", ecolor="#6B6B6B", capsize=4, markersize=7,
)
ax.axvline(x=1, color="#6B6B6B", linestyle="--", alpha=0.7, label="RR = 1 (no effect)")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(rr_sorted["factor"])
ax.set_xlabel("Risk Ratio (95% CI)")
ax.set_title("Crude Risk Ratio by Factor (Forest Plot)")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

| Step | Skill learned | Python tool |
|------|-----------|------------|
| 2×2 table | Building a contingency table | `pd.crosstab()` |
| RR | Computing and interpreting the risk ratio | `risk_ratio()` |
| OR | Computing the odds ratio, comparing with RR | `odds_ratio()` |
| 95% CI | Confidence intervals for RR and OR | Katz / Woolf method |
| Chi-square test | Test of independence | `chi2_contingency()` |
| Fisher's exact test | Small-sample alternative | `fisher_exact()` |
| Multi-factor scan | Loop + summary + forest plot | `for` + `DataFrame` + `matplotlib` |

### When Do You Use RR? When Do You Use OR?

| Situation | Which to use | Why |
|------|--------|--------|
| Cohort study (like this outbreak investigation) | **RR** | Complete denominators, so you can compute risk directly |
| Case-control study | **OR** | No complete denominators, can't compute risk |
| Logistic regression output | **OR** | The model output is log-odds |
| Any study design of a rare disease | Either | When rare, OR ≈ RR |
| High attack rate (> 10%) | **RR** | The OR systematically overestimates the risk multiple |
| Vaccine effectiveness (cohort) | VE = 1−**RR** | Clinical trials, outbreak investigations |
| Vaccine effectiveness (case-control) | VE = 1−**OR** | Large-scale post-marketing surveillance |

### Important Reminders

1. **A crude RR / OR is just a preliminary clue**. Shower use has a high-looking effect size, but it may be affected by **confounding**—like a burger that looks huge but is really just propped up by lettuce. Ch05 will handle this with **stratified analysis** and the **Mantel-Haenszel method**.

2. **Statistical significance ≠ causation**. Testing 8 factors, by chance alone you could get ~0.4 "false positives" (at α=0.05). Finding an association is just the starting point.

3. **Ch06** will use **logistic regression** to adjust for multiple factors at once and compute an adjusted OR (remember: the regression's native output is an OR, and you have to interpret it carefully when the attack rate is high).